In [12]:
# === IMPORTAÇÕES ===
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score

#!pip install -U spacy
#!python -m spacy download pt_core_news_sm
import spacy
import webbrowser
import pickle
from random import choice

nlp = spacy.load("pt_core_news_sm")

In [14]:
# Sentimental Analysis Algorithm - Algoritmo de Análise de Sentimento

# === 1. PRÉ-PROCESSAMENTO DE TEXTO ===
def preprocess_text(text):
    """
    Converte texto para minúsculas, tokeniza, lematiza,
    remove stopwords, pontuação e espaços.
    Retorna lista de tokens.
    """
    doc = nlp(text.lower())
    tokens = [
        token.lemma_ for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space
    ]
    return tokens

# Dicionário de sinônimos para data augmentation
SYNONYMS = {
    "bom": ["ótimo", "excelente"],
    "ruim": ["péssimo", "terrível"],
    # Adicione mais sinônimos conforme necessidade
}

def augment_tokens(tokens, n_swaps=1):
    """Substitui até n_swaps tokens por sinônimos."""
    for _ in range(n_swaps):
        idxs = [i for i, t in enumerate(tokens) if t in SYNONYMS]
        if not idxs:
            break
        ix = choice(idxs)
        tokens[ix] = choice(SYNONYMS[tokens[ix]])
    return tokens

# === 2. CARGA E TRATAMENTO DOS DADOS ===
def load_data(path, augment=False):
    """
    Lê CSV, aplica pré-processamento de texto e opcionalmente
    data augmentation duplicando metade dos dados com sinônimos.
    """
    df = pd.read_csv(path, encoding='utf-8')
    df.dropna(inplace=True)
    df['tokens'] = df['texto'].apply(preprocess_text)

    if augment:
        # Duplica 50% dos registros para augmenter
        extra = df.sample(frac=0.5, random_state=42).copy()
        extra['tokens'] = extra['tokens'].apply(lambda t: augment_tokens(t.copy()))
        df = pd.concat([df, extra], ignore_index=True)

    return df

# Cria vocabulário baseado na frequência mínima dos tokens
def build_vocab(token_lists, min_freq=1):
    freq = {}
    for tokens in token_lists:
        for token in tokens:
            freq[token] = freq.get(token, 0) + 1
    vocab = {word: i + 1 for i, (word, count) in enumerate(freq.items()) if count >= min_freq}
    vocab['<UNK>'] = 0  # Token desconhecido
    return vocab

# Codifica tokens para índices do vocabulário
def encode_tokens(tokens, vocab, max_len=50):
    indices = [vocab.get(t, 0) for t in tokens]
    return indices[:max_len] + [0] * max(0, max_len - len(indices))

# === 3. CLASSE DE DATASET ===
class TextDataset(Dataset):
    def __init__(self, df, vocab, label_encoder):
        self.inputs = [encode_tokens(t, vocab) for t in df['tokens']]
        self.labels = label_encoder.transform(df['sentimento'])

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx]), torch.tensor(self.labels[idx])

# === 4. DEFS DOS MODELOS ===
# MLP (Perceptron Multicamadas)
class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        pooled = embedded.mean(dim=1)
        dropped = self.dropout(pooled)
        return self.fc(dropped)

# LSTM Unidirecional
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        dropped = self.dropout(hidden[-1])
        return self.fc(dropped)

# Bi-LSTM Bidirecional
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        hidden_cat = torch.cat((hidden[-2], hidden[-1]), dim=1)
        dropped = self.dropout(hidden_cat)
        return self.fc(dropped)

# === 5. FUNÇÃO DE TREINO E VALIDAÇÃO ===
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device, label_encoder):
    """
    Treina o modelo, valida por época, gera relatório HTML e retorna métricas.
    """
    model.to(device)
    all_labels, all_preds = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val = x_val.to(device)
                outputs = model(x_val)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(y_val.numpy())

        epoch_acc = accuracy_score(all_labels, all_preds)
        print(f"Epoch {epoch}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, Acc: {epoch_acc:.2f}")

    # Métricas finais: cálcula acurácia e F1-macro
    final_accuracy = accuracy_score(all_labels, all_preds)
    final_f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    # Gera relatório em dict e matriz de confusão
    report_dict = classification_report(
      all_labels, all_preds,
      target_names=label_encoder.classes_,
      output_dict=True,
      zero_division=0
      )
    conf_matrix = confusion_matrix(all_labels, all_preds)

    # HTML report
    html = classification_report_to_html(
      report_dict,
      label_encoder.classes_,
      final_accuracy,
      conf_matrix
      )

    html_path = f"resultado_{model.__class__.__name__}.html"
    with open(html_path, 'w', encoding='utf-8') as f:
      f.write(html)
      print(f"Relatório salvo em {html_path}")
    webbrowser.open(f"file://{os.path.abspath(html_path)}")

    # Salva modelo treinado
    torch.save(
      model.state_dict(),
      f"modelo_{model.__class__.__name__.lower()}.pth"
      )
    print(f"Modelo {model.__class__.__name__} salvo.")

    # Retorna as métricas para comparação
    return final_accuracy, final_f1_macro

# === UTILitário: HTML REPORT ===
def classification_report_to_html(report_dict, class_names, accuracy, conf_matrix):
    html = f"""
    <html><head><title>Relatório de {class_names}</title></head><body>
    <h1>Relatório de Classificação</h1>
    <h2>Acurácia Final: {accuracy:.2f}</h2>
    <h3>Métricas por Classe</h3>
    <table border='1' cellpadding='8'><tr><th>Classe</th><th>Precision</th><th>Recall</th><th>F1-score</th><th>Support</th></tr>
    """
    for label in class_names:
        m = report_dict[label]
        html += f"<tr><td>{label}</td><td>{m['precision']:.2f}</td><td>{m['recall']:.2f}</td><td>{m['f1-score']:.2f}</td><td>{m['support']}</td></tr>"
    html += "</table><h3>Matriz de Confusão</h3><table border='1' cellpadding='8'><tr><th></th>"
    for l in class_names:
        html += f"<th>{l}</th>"
    html += "</tr>"
    for i, row in enumerate(conf_matrix):
        html += f"<tr><th>{class_names[i]}</th>"
        for v in row:
            html += f"<td>{v}</td>"
        html += "</tr>"
    html += "</table></body></html>"
    return html

# === 6. EXECUÇÃO PRINCIPAL ===
def main():
    filepath = "posts_new.csv"  # Ajuste se necessário
    # Carrega dados com augmentation ativado
    df = load_data(filepath, augment=True)

    # Vocab e label encoder
    vocab = build_vocab(df['tokens'])
    label_encoder = LabelEncoder()
    label_encoder.fit(df['sentimento'])

    dataset = TextDataset(df, vocab, label_encoder)
    # Divisão 80/20
    from sklearn.model_selection import train_test_split
    idx = list(range(len(dataset)))
    train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)
    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=8, shuffle=True)
    val_loader   = DataLoader(Subset(dataset, val_idx), batch_size=8)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    criterion = nn.CrossEntropyLoss()

    # Inicializa modelos
    models = {
        'MLP': SentimentClassifier(len(vocab), embed_dim=100, num_classes=len(label_encoder.classes_), dropout=0.4),
        'LSTM': LSTMClassifier(len(vocab), embed_dim=100, hidden_dim=64, num_classes=len(label_encoder.classes_), dropout=0.4),
        'BiLSTM': BiLSTMClassifier(len(vocab), embed_dim=100, hidden_dim=64, num_classes=len(label_encoder.classes_), dropout=0.4),
    }

    # Treina e coleta resultados
    results = {}
    for name, model in models.items():
        print(f"\n>>> Treinando {name}")
        optimizer = torch.optim.RMSprop(model.parameters(), lr=0.01, alpha=0.9)
        acc, f1 = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=32, device=device, label_encoder=label_encoder)
        results[name] = {'accuracy': acc, 'f1_macro': f1}

    # Exibe comparativo
    print("\n=== Comparativo Final ===")
    print(pd.DataFrame(results).T)

    # Salva dependências
    with open('vocab.pkl', 'wb') as f:
        pickle.dump(vocab, f)
    with open('label_encoder.pkl', 'wb') as f:
        pickle.dump(label_encoder, f)
    print("Vocabulário e LabelEncoder salvos.")

if __name__ == '__main__':
    main()


>>> Treinando MLP
Epoch 1/32, Loss: 1.4944, Acc: 0.33
Epoch 2/32, Loss: 1.2109, Acc: 0.22
Epoch 3/32, Loss: 1.2311, Acc: 0.22
Epoch 4/32, Loss: 1.3193, Acc: 0.33
Epoch 5/32, Loss: 1.1650, Acc: 0.44
Epoch 6/32, Loss: 1.0080, Acc: 0.22
Epoch 7/32, Loss: 1.3081, Acc: 0.44
Epoch 8/32, Loss: 1.1002, Acc: 0.22
Epoch 9/32, Loss: 1.2726, Acc: 0.22
Epoch 10/32, Loss: 1.2561, Acc: 0.33
Epoch 11/32, Loss: 1.2146, Acc: 0.33
Epoch 12/32, Loss: 1.0467, Acc: 0.33
Epoch 13/32, Loss: 1.3942, Acc: 0.67
Epoch 14/32, Loss: 1.2207, Acc: 0.67
Epoch 15/32, Loss: 1.1854, Acc: 0.44
Epoch 16/32, Loss: 1.1402, Acc: 0.44
Epoch 17/32, Loss: 1.1284, Acc: 0.44
Epoch 18/32, Loss: 1.0389, Acc: 0.78
Epoch 19/32, Loss: 1.0605, Acc: 0.67
Epoch 20/32, Loss: 1.3091, Acc: 0.67
Epoch 21/32, Loss: 1.0044, Acc: 0.44
Epoch 22/32, Loss: 0.9735, Acc: 0.56
Epoch 23/32, Loss: 1.1934, Acc: 0.78
Epoch 24/32, Loss: 0.9832, Acc: 0.44
Epoch 25/32, Loss: 0.9520, Acc: 0.44
Epoch 26/32, Loss: 1.0599, Acc: 0.44
Epoch 27/32, Loss: 0.8853, A